# house prices in sri lanka

goal is to predict the asking price of a house for sale in sri lanka. data is the ikman.lk property ads dataset from kaggle (ivantha).

In [1]:
import os, io, glob, ast, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)

## 1. getting the data

downloaded with kagglehub into `../data/raw`. that folder is ignored by git, only the cleaned file gets saved in the repo.

In [2]:
os.environ['KAGGLEHUB_CACHE'] = '../data/raw'
import kagglehub

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    kagglehub.dataset_download('ivantha/sri-lanka-property-ads-dataset')

csv_path = glob.glob('../data/raw/**/properties.csv', recursive=True)[0]
raw = pd.read_csv(csv_path)
print(raw.shape)

(203874, 26)


In [3]:
raw[['category', 'type']].value_counts().head(8)

category                     type    
Land For Sale                for_sale    82123
Houses For Sale              for_sale    51890
Land                         for_sale    27046
House Rentals                for_rent    14754
Apartment Rentals            for_rent     7164
Apartments For Sale          for_sale     5177
Commercial Property Rentals  for_rent     4415
Room & Annex Rentals         for_rent     4373
Name: count, dtype: int64

the file has everything people post under property: land, houses, apartments, rentals and commercial. land is often priced per perch and rentals per month, so those prices cannot sit in one target column. i keep only **houses for sale** (52,193 ads), where the price is the full asking price in rupees.

In [4]:
houses = raw[raw['category'] == 'Houses For Sale'].copy()
print(houses.shape)
houses[['price', 'details', 'location', 'properties']].head()

(52193, 26)


,price,details,location,properties
4,"Rs 55,000,000","Bedrooms: 5, Bathrooms: 4",Kadawatha,"{'Address': 'Pinthaliya Road, Kadawatha', 'Bed..."
9,"Rs 80,000,000","Bedrooms: 4, Bathrooms: 4",Dehiwala,"{'Address': 'kadawatha Road', 'Bedrooms': '4',..."
21,"Rs 125,000,000","Bedrooms: 4, Bathrooms: 2",Mount Lavinia,"{'Address': 'Off Temple Road, Mount Lavinia', ..."
22,"Rs 58,000,000","Bedrooms: 4, Bathrooms: 3",Malabe,"{'Bedrooms': '4', 'Bathrooms': '3', 'House siz..."
24,"Rs 22,000,000","Bedrooms: 4, Bathrooms: 3",Kotte,"{'Address': 'Kotte', 'Bedrooms': '4', 'Bathroo..."


## 2. parsing the text columns

the details are inside a dictionary string in `properties`. price is text like `Rs 38,000,000` and the district is inside the `area` string. i pull out price, bedrooms, bathrooms and house size for now.

In [5]:
def parse_dict(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return {}

props = houses['properties'].apply(parse_dict)

def first_number(text):
    if pd.isna(text):
        return np.nan
    return pd.to_numeric(str(text).replace(',', '').split(' ')[0], errors='coerce')

df = pd.DataFrame({
    'price': houses['price'].str.replace(r'[^0-9]', '', regex=True).pipe(pd.to_numeric, errors='coerce'),
    'bedrooms': props.apply(lambda d: first_number(d.get('Bedrooms'))),
    'bathrooms': props.apply(lambda d: first_number(d.get('Bathrooms'))),
    'house_size_sqft': props.apply(lambda d: first_number(d.get('House size'))),
    'location': houses['location'].str.strip().str.title(),
    'district': houses['area'].apply(lambda s: parse_dict(s).get('name')),
}).reset_index(drop=True)
df.head()

,price,bedrooms,bathrooms,house_size_sqft,location,district
0,55000000.0,5.0,4.0,3000.0,Kadawatha,Gampaha
1,80000000.0,4.0,4.0,3000.0,Dehiwala,Colombo
2,125000000.0,4.0,2.0,5600.0,Mount Lavinia,Colombo
3,58000000.0,4.0,3.0,4000.0,Malabe,Colombo
4,22000000.0,4.0,3.0,2200.0,Kotte,Colombo
